### Exploração dos dados Parquet
Vamos ler os arquivos Parquet da camada Bronze e fazer consultas iniciais.

In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()

In [3]:
base = '../data/silver/'

# Ler um parquet
parquet_file = base + 'bnds_silver.parquet'
df = con.execute(f"SELECT * FROM '{parquet_file}' LIMIT 5").fetchdf()
df.head()

,razao_social_cliente,cnpj_cliente,descricao_projeto,sigla_uf,nome_municipio,id_municipio,id_contrato,data_contratacao,valor_contratado,valor_desembolsado,...,ano_contratacao,mes_contratacao,prazo_total_meses,grupo_taxa_juros,grupo_taxa_juros_ordinal,flag_apoio_direto,flag_reembolsavel,flag_cliente_publico,qtd_palavras_descricao,porte_cliente_ordinal
0,COPACOL-COOPERATIVA AGROINDUSTRIAL CONSOLATA,76093731000190.0,INVESTIMENTOS EM AMPLIACAO DA CAPACIDADE DE RE...,PR,SEM MUNICIPIO,NAO INFORMADO,9203361,2009-06-30,35900000.0,35900000.0,...,2009,6,108,MEDIA,2,0,1,0,44,4
1,LAR COOPERATIVA AGROINDUSTRIAL,77752293000198.0,DIRETA - AMPLICACAO DE LINHA DE ABATE DE FRANG...,PR,MATELANDIA,4115606.0,9203571,2009-06-29,66998000.0,66998000.0,...,2009,6,108,MEDIA,2,1,1,0,24,4
2,LAR COOPERATIVA AGROINDUSTRIAL,77752293000198.0,DIRETA - AMPLICACAO DE LINHA DE ABATE DE FRANG...,MS,SEM MUNICIPIO,NAO INFORMADO,9203581,2009-06-30,20897000.0,20897000.0,...,2009,6,108,MEDIA,2,0,1,0,24,4
3,COCARI - COOPERATIVA AGROPECUARIA E INDUSTRIAL,78956968000183.0,SUPLEMENTACAO DE RECURSOS A IMPLANTACAO DE UM ...,PR,MANDAGUARI,4114203.0,9204441,2009-06-30,37000000.0,37000000.0,...,2009,6,120,MEDIA,2,0,1,0,45,4
4,COCARI - COOPERATIVA AGROPECUARIA E INDUSTRIAL,78956968000183.0,SUPLEMENTACAO DE RECURSOS A IMPLANTACAO DE UM ...,PR,MANDAGUARI,4114203.0,9204451,2009-10-09,50000000.0,50000000.0,...,2009,10,120,MEDIA,2,0,1,0,45,4


In [4]:
# Mostrar colunas do parquet
schema = con.execute(f"DESCRIBE SELECT * FROM '{parquet_file}'").fetchdf()
schema

,column_name,column_type,null,key,default,extra
0,razao_social_cliente,VARCHAR,YES,None,None,None
1,cnpj_cliente,VARCHAR,YES,None,None,None
2,descricao_projeto,VARCHAR,YES,None,None,None
3,sigla_uf,VARCHAR,YES,None,None,None
4,nome_municipio,VARCHAR,YES,None,None,None
5,id_municipio,VARCHAR,YES,None,None,None
6,id_contrato,BIGINT,YES,None,None,None
7,data_contratacao,TIMESTAMP,YES,None,None,None
8,valor_contratado,DOUBLE,YES,None,None,None
9,valor_desembolsado,DOUBLE,YES,None,None,None


In [5]:
# Consultas iniciais

# Quantidade de linhas
total_linhas = con.execute(f"SELECT COUNT(*) AS total_linhas FROM '{parquet_file}'").fetchdf()
total_linhas

,total_linhas
0,23483


### Consulta Diretorios CNAE

In [6]:
base_path = '../data/silver/'

# Ler um parquet
parquet_file = base_path + 'br_bd_diretorios_brasil_cnae_2.parquet'
df = con.execute(f"SELECT * FROM '{parquet_file}' LIMIT 5").fetchdf()
df.head()

,subclasse,descricao_subclasse,classe,descricao_classe,grupo,descricao_grupo,divisao,descricao_divisao,secao,descricao_secao,indicador_cnae_2_0,indicador_cnae_2_1,indicador_cnae_2_2,indicador_cnae_2_3
0,9700500,Serviços domésticos,97005,Serviços domésticos,970,Serviços domésticos,97,Serviços Domésticos,T,Serviços Domésticos,1,1,1,1
1,9900800,Organismos internacionais e outras instituiçõe...,99008,Organismos internacionais e outras instituiçõe...,990,Organismos internacionais e outras instituiçõe...,99,Organismos Internacionais E Outras Instituiçõe...,U,Organismos Internacionais E Outras Instituiçõe...,1,1,1,0
2,111301,Cultivo de arroz,1113,Cultivo de cereais,11,Produção de lavouras temporárias,1,"Agricultura, Pecuária E Serviços Relacionados",A,"Agricultura, Pecuária, Produção Florestal, Pes...",1,1,1,1
3,111399,Cultivo de outros cereais não especificados an...,1113,Cultivo de cereais,11,Produção de lavouras temporárias,1,"Agricultura, Pecuária E Serviços Relacionados",A,"Agricultura, Pecuária, Produção Florestal, Pes...",1,1,1,1
4,111302,Cultivo de milho,1113,Cultivo de cereais,11,Produção de lavouras temporárias,1,"Agricultura, Pecuária E Serviços Relacionados",A,"Agricultura, Pecuária, Produção Florestal, Pes...",1,1,1,1


### Consulta Diretorios Municípios

In [7]:
base_path = '../lake/silver/'

# Ler um parquet
parquet_file = base_path + 'br_bd_diretorios_brasil_municipio.parquet'
df = con.execute(f"SELECT * FROM '{parquet_file}' LIMIT 5").fetchdf()
df.head()

IOException: IO Error: No files found that match the pattern "../lake/silver/br_bd_diretorios_brasil_municipio.parquet"

### Join com descrição da divisão CNAE
Vamos unir a tabela de operações com o diretório de CNAE para trazer a descrição da divisão.

In [ ]:
base_operacoes = '../silver/br_bndes_operacoes.parquet'
base_cnae = '../silver/br_bd_diretorios_brasil_cnae_2.parquet'
base_municipio = '../silver/br_bd_diretorios_brasil_municipio.parquet'

query = f"""
SELECT op.*, cnae.descricao_divisao, mn.sigla_uf, mn.nome_regiao FROM '{base_operacoes}' AS op
LEFT JOIN '{base_cnae}' AS cnae
    ON op.divisao_cnae = cnae.divisao
LEFT JOIN '{base_municipio}' AS mn
    ON op.id_municipio = mn.id_municipio
LIMIT 10
"""

resultado = con.execute(query).fetchdf()
resultado.head()